# 01 — Project overview + reproducibility plan (phases + artifacts)

A map of the repo, what each phase produces, and Mermaid diagrams for your paper/README.


In [3]:
from __future__ import annotations

import os, sys, subprocess, json, re
from pathlib import Path
from datetime import datetime

REPO_ROOT = Path.cwd().parent
print("REPO_ROOT:", REPO_ROOT)

def sh(cmd: str, check: bool=True) -> None:
    """Run a shell command (prints it first)."""
    print("\n▶", cmd)
    subprocess.run(cmd, shell=True, check=check)

def pick_first_existing(*cands: str) -> Path:
    for c in cands:
        p = Path(c)
        if p.exists():
            return p
    return Path(cands[0])

def require_exists(p: Path, what: str="path") -> Path:
    if not p.exists():
        raise FileNotFoundError(f"Missing {what}: {p}")
    return p

def newest_path(glob_pat: str) -> Path | None:
    paths = list(REPO_ROOT.glob(glob_pat))
    if not paths:
        return None
    paths.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return paths[0]

def show_tree(root: Path, max_lines: int=200) -> None:
    i = 0
    for p in sorted(root.rglob("*")):
        if i >= max_lines:
            print("... (truncated)")
            return
        if p.is_dir():
            continue
        rel = p.relative_to(root)
        print(rel)
        i += 1

print("Top-level folders:")
for name in sorted([p.name for p in REPO_ROOT.iterdir() if p.is_dir()]):
    print(" -", name)


REPO_ROOT: /Users/ameerfiras/REDNET-ML
Top-level folders:
 - .git
 - architecture
 - archive
 - catboost_info
 - cfg
 - data
 - deployment
 - detection_models
 - notebooks
 - rednet-risk-viewer
 - report
 - runs
 - scripts
 - src
 - test
 - training


## 1.1 Data + training pipeline (Mermaid)


In [4]:

mermaid_data = r'''
flowchart TD
  A[AOI + Plant AOIs] --> B[Sentinel-2 chipping\ns2_chip_8day.py]
  B --> C[Compute chip indices\ns2_compute_chip_indices.py]
  D[OBPG filelists\nmake_obpg_filelists_8d.py] --> E[OBDAAC download\nobdaac_download.py]
  E --> F[Append MODIS features\nappend_modis_features_8d.py]
  C --> F
  F --> G[Label mining + non-leaky train\nscripts/HAB/preparation/*]
  G --> H[Detector datasets + training\nsrc/torchvision_det/*]
  G --> I[Tabular fusion training\nscripts/fusion/*]
  H --> J[Detector scores on chips\nrun_detectors_on_chips.py]
  J --> K[Rerun fusion w/ detector scores\nrerun_fusion_with_detectors.py]
  K --> L[Risk aggregation + Kepler package]
'''
print(mermaid_data)



flowchart TD
  A[AOI + Plant AOIs] --> B[Sentinel-2 chipping\ns2_chip_8day.py]
  B --> C[Compute chip indices\ns2_compute_chip_indices.py]
  D[OBPG filelists\nmake_obpg_filelists_8d.py] --> E[OBDAAC download\nobdaac_download.py]
  E --> F[Append MODIS features\nappend_modis_features_8d.py]
  C --> F
  F --> G[Label mining + non-leaky train\nscripts/HAB/preparation/*]
  G --> H[Detector datasets + training\nsrc/torchvision_det/*]
  G --> I[Tabular fusion training\nscripts/fusion/*]
  H --> J[Detector scores on chips\nrun_detectors_on_chips.py]
  J --> K[Rerun fusion w/ detector scores\nrerun_fusion_with_detectors.py]
  K --> L[Risk aggregation + Kepler package]



## 1.2 Inference pipeline (Mermaid)


In [5]:

mermaid_infer = r'''
flowchart TD
  A[Select AOI + time range] --> B[s2_chip_8day.py\n(month loop, 8-day windows)]
  B --> C[s2_compute_chip_indices.py]
  C --> D[MODIS on-demand download\nobdaac_download.py]
  D --> E[append_modis_features_8d.py]
  E --> F[Tabular model predict -> hab_prob]
  F --> G[Write per-month inference.csv]
  G --> H[Run detectors (optional)]
  H --> I[Rerun fusion + merge all months]
'''
print(mermaid_infer)



flowchart TD
  A[Select AOI + time range] --> B[s2_chip_8day.py\n(month loop, 8-day windows)]
  B --> C[s2_compute_chip_indices.py]
  C --> D[MODIS on-demand download\nobdaac_download.py]
  D --> E[append_modis_features_8d.py]
  E --> F[Tabular model predict -> hab_prob]
  F --> G[Write per-month inference.csv]
  G --> H[Run detectors (optional)]
  H --> I[Rerun fusion + merge all months]

